Notebook 2b: Dictionary lookup

This notebook builds the non-GenAI baseline: a plain dictionary lookup. A word list is built from the gold
text of the BLN600 excerpts outside the sample, and every damaged word is replaced by the list entry whose
spelling is closest to its damaged form. There is no context and no model. It is the pure "confusion" side of
"context beats confusion" (Evershed and Fitch, 2014): BERT and the LLM add the context, so this baseline shows
what the context is worth.

**Input.** The frozen damaged dataset `data/processed/corrupted_v2.jsonl` (hash checked against
`runs/corrupted_v2.sha256`), BLN600 in `data/raw/BLN600/`, and `configs/lexicon.yaml`.

**Output.** The stored predictions in `runs/preds/lexicon_<version>.jsonl`, scored in notebook 4 together
with the other methods.

`SPLIT` chooses the rows. The test split stays closed until `ALLOW_TEST` is set and `frozen: true` is set in
`configs/lexicon.yaml`. Nothing is tuned here, so the dev run is only a check.

In [1]:
import inspect

import pandas as pd

from blnrepair.data import ROOT, load_config, load_gold
from blnrepair.freeze import load_frozen
from blnrepair.lexicon import LexiconLookup, build_lexicon, lexicon_version, load_lexicon_config
from blnrepair.preds import load_preds, pred_path, run_method, slot_exact
from blnrepair.slots import build_slots, slot_view

pd.set_option("display.max_colwidth", None)
config, lex_cfg = load_config(), load_lexicon_config()
rows = load_frozen("v2")  # stops if the file hash does not match runs/corrupted_v2.sha256

SPLIT, ALLOW_TEST = "dev", False
if SPLIT == "test":
    assert ALLOW_TEST and lex_cfg["frozen"], "the test split is closed: set ALLOW_TEST and freeze configs/lexicon.yaml"

The word list

The list holds every word of the 443 excerpts that are not in the sample, so no test or dev sentence
contributes a word. A word is taken without its edge punctuation, with the same rule as the slot view
(inner hyphens and apostrophes stay, so `Market-place` is one entry), and case is kept. Each entry keeps its
count, which breaks ties below.

In [2]:
sample_docs = {r["doc_id"] for r in rows}
gold = load_gold(ROOT / config["paths"]["raw_dir"])
counts = build_lexicon(gold, sample_docs)
print(f"{len(gold) - len(sample_docs)} excerpts outside the sample, {len(counts):,} distinct words, "
      f"{sum(counts.values()):,} words in total")

443 excerpts outside the sample, 16,469 distinct words, 213,237 words in total


The lookup

For each slot with visible letters, the damaged form is compared with every entry of the list by edit
distance (insertions, deletions and substitutions of single characters, case-sensitive, the same measure
BERT's reranking uses). The closest entry wins; among equally close entries, the most frequent one. A dropped
word `⟨?⟩` has no letters to look up, so the lookup leaves it empty: the word stays missing and counts as
wrong.

In [3]:
print(inspect.getsource(LexiconLookup.nearest))

    def nearest(self, core):
        """(entry, edit distance, number of entries at that distance) for one damaged core."""
        dist = cdist([core], self.words, scorer=Levenshtein.distance, workers=-1)[0]
        best = int(dist.min())
        i = int(dist.argmin())  # first minimum = the most frequent of the closest entries
        return self.words[i], best, int((dist == best).sum())



Running the lookup

The lookup runs on every row of the split that has slots, through the same prediction store as the other
methods. It takes a few seconds for a whole split.

In [4]:
lookup = LexiconLookup(counts)
version = lexicon_version(lex_cfg, SPLIT)
split_rows = [r for r in rows if r["split"] == SPLIT]
added = run_method(split_rows, lookup, "lexicon", version, "dictionary lookup", allow_test=ALLOW_TEST)
stored = load_preds(pred_path("lexicon", version))
print(f"lexicon {version}: {len(stored)} rows stored ({len(added)} new in this run)")

lexicon v1-dev: 100 rows stored (100 new in this run)


Examples from dev

A few dev repairs at the one-word level, with the damaged form, the lookup's answer and the gold word. The
examples always come from dev, whatever `SPLIT` is, so no test result is shown here. The share of exact
words per level is a sanity check, not a result; the scores are in notebook 4.

In [5]:
dev_rows = [r for r in rows if r["split"] == "dev" and r["k"]]
dev_preds = {(p["id"], p["severity"]): p for p in load_preds(pred_path("lexicon", lexicon_version(lex_cfg, "dev")))}
one_word = [r for r in dev_rows if r["severity"] == "1w"][:8]
pd.DataFrame({"damaged": [r["ops_per_word"][0]["out"] for r in one_word],
              "lookup": [dev_preds[(r["id"], "1w")]["pred_words"][0] for r in one_word],
              "gold": [r["ops_per_word"][0]["gold"] for r in one_word]})

,damaged,lookup,gold
0,"£1,",1,"£2,"
1,Raiiy-ststion,manifestation,Railway-station
2,Wiiliani,William,William
3,"NIaruet-plece,",Dorset-place,"Market-place,"
4,"l8,",18,"18,"
5,Suney,money,Sunday
6,Hainos,Haines,Haines
7,"Wrighb,",Wright,"Wright,"


In [6]:
pd.Series({level: sum(slot_exact(dev_preds[(r["id"], level)]["pred_words"], r)
                      for r in dev_rows if r["severity"] == level) / sum(r["severity"] == level for r in dev_rows)
           for level in ["1w", "10", "25", "50", "75"]}, name="exact words, dev (sanity check)").round(3)

1w    0.450
10    0.463
25    0.529
50    0.610
75    0.588
Name: exact words, dev (sanity check), dtype: float64